# CartIQ — Phase 1: Data Preparation
**CS Machine Learning Course Project — Group 3**

This notebook downloads the Instacart dataset, cleans it, merges tables, and constructs supervised labels for the reorder prediction task.

In [ ]:
# Install dependencies
!pip install -q kagglehub pandas numpy scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
pd.set_option('display.max_columns', 50)
print('Libraries loaded.')

## 1.1 Download Instacart Dataset

In [ ]:
import kagglehub

path = kagglehub.dataset_download('yasserh/instacart-online-grocery-basket-analysis-dataset')
print('Dataset downloaded to:', path)

# List files
for f in sorted(os.listdir(path)):
    fpath = os.path.join(path, f)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f'  {f:45s} {size_mb:8.2f} MB')

## 1.2 Load CSV Tables

In [ ]:
orders = pd.read_csv(os.path.join(path, 'orders.csv'))
products = pd.read_csv(os.path.join(path, 'products.csv'))
aisles = pd.read_csv(os.path.join(path, 'aisles.csv'))
departments = pd.read_csv(os.path.join(path, 'departments.csv'))
order_products_prior = pd.read_csv(os.path.join(path, 'order_products__prior.csv'))
order_products_train = pd.read_csv(os.path.join(path, 'order_products__train.csv'))

print(f'orders:                {orders.shape}')
print(f'products:              {products.shape}')
print(f'aisles:                {aisles.shape}')
print(f'departments:           {departments.shape}')
print(f'order_products_prior:  {order_products_prior.shape}')
print(f'order_products_train:  {order_products_train.shape}')

In [ ]:
print('=== orders ===')
display(orders.head())
print('\n=== products ===')
display(products.head())
print('\n=== order_products_prior ===')
display(order_products_prior.head())

## 1.3 Handle Missing Values

In [ ]:
print('Missing values in orders:')
print(orders.isnull().sum())
print(f'\ndays_since_prior_order NaN count: {orders["days_since_prior_order"].isnull().sum()}')
print('(These are first-ever orders for each user — expected.)')

# Fill NaN with 0 for first orders
orders['days_since_prior_order'] = orders['days_since_prior_order'].fillna(0)
print(f'\nAfter fill: NaN count = {orders["days_since_prior_order"].isnull().sum()}')

## 1.4 Merge Tables

In [ ]:
# Enrich products with aisle and department names
products = products.merge(aisles, on='aisle_id', how='left')
products = products.merge(departments, on='department_id', how='left')
print(f'Products enriched: {products.shape}')
display(products.head())

In [ ]:
# Merge prior order products with orders and products
prior = order_products_prior.merge(orders, on='order_id', how='left')
prior = prior.merge(products[['product_id', 'product_name', 'aisle', 'department']], on='product_id', how='left')
print(f'Prior orders merged: {prior.shape}')
display(prior.head())

## 1.5 Construct Supervised Labels
For each (user, product) pair in the train set, the label is `reordered` ∈ {0, 1}.

In [ ]:
# Merge train order products with orders
train_orders = orders[orders['eval_set'] == 'train']
train_data = order_products_train.merge(train_orders[['order_id', 'user_id']], on='order_id', how='left')

print(f'Train data shape: {train_data.shape}')
print(f'\nLabel distribution:')
print(train_data['reordered'].value_counts())
print(f'\nReorder rate: {train_data["reordered"].mean():.4f}')

# Class imbalance
neg = (train_data['reordered'] == 0).sum()
pos = (train_data['reordered'] == 1).sum()
print(f'\nClass imbalance ratio (neg:pos): {neg/pos:.1f}:1')

## 1.6 Dataset Statistics & Visualization

In [ ]:
print(f'Total unique users:    {orders["user_id"].nunique():,}')
print(f'Total unique products: {products["product_id"].nunique():,}')
print(f'Total orders:          {orders.shape[0]:,}')
print(f'Total prior items:     {order_products_prior.shape[0]:,}')
print(f'Total train items:     {order_products_train.shape[0]:,}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Orders per user
user_orders = orders.groupby('user_id').size()
axes[0, 0].hist(user_orders, bins=50, color='#3b82f6', edgecolor='white', alpha=0.8)
axes[0, 0].set_title('Orders Per User', fontweight='bold')
axes[0, 0].set_xlabel('Number of Orders')
axes[0, 0].set_ylabel('Users')

# Order day of week
axes[0, 1].bar(range(7), orders['order_dow'].value_counts().sort_index(),
               color='#10b981', edgecolor='white')
axes[0, 1].set_xticks(range(7))
axes[0, 1].set_xticklabels(['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat'])
axes[0, 1].set_title('Orders by Day of Week', fontweight='bold')

# Order hour of day
axes[1, 0].bar(range(24), orders['order_hour_of_day'].value_counts().sort_index(),
               color='#f59e0b', edgecolor='white')
axes[1, 0].set_title('Orders by Hour of Day', fontweight='bold')
axes[1, 0].set_xlabel('Hour')

# Days since prior order
axes[1, 1].hist(orders[orders['days_since_prior_order'] > 0]['days_since_prior_order'],
                bins=30, color='#ef4444', edgecolor='white', alpha=0.8)
axes[1, 1].set_title('Days Since Prior Order', fontweight='bold')
axes[1, 1].set_xlabel('Days')

plt.tight_layout()
plt.savefig('dataset_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top departments and aisles
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

dept_counts = prior['department'].value_counts().head(10)
axes[0].barh(dept_counts.index[::-1], dept_counts.values[::-1], color='#6366f1')
axes[0].set_title('Top 10 Departments', fontweight='bold')

aisle_counts = prior['aisle'].value_counts().head(15)
axes[1].barh(aisle_counts.index[::-1], aisle_counts.values[::-1], color='#ec4899')
axes[1].set_title('Top 15 Aisles', fontweight='bold')

plt.tight_layout()
plt.savefig('top_categories.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.7 Save Processed Data

In [ ]:
os.makedirs('data', exist_ok=True)

orders.to_parquet('data/orders.parquet', index=False)
products.to_parquet('data/products.parquet', index=False)
prior.to_parquet('data/prior.parquet', index=False)
train_data.to_parquet('data/train_labels.parquet', index=False)

print('Saved:')
for f in os.listdir('data'):
    size = os.path.getsize(os.path.join('data', f)) / 1e6
    print(f'  data/{f:30s} {size:8.2f} MB')